In [1]:
from secedgar.cik_lookup import get_cik_map
import requests
import numpy as np
import pandas as pd
import time
import io

USER_AGENT = "Academic Research Project (gig-economy-capstone@umich.edu)"

c:\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


In [2]:
upwk_cik = '1627475'
cik_padded = str(upwk_cik).zfill(10)

cik_padded = upwk_cik.zfill(10)
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik_padded}.json"

headers = {
    'User-Agent': USER_AGENT,
    'Accept-Encoding': 'gzip, deflate'
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
data = response.json()

results = []
facts = data.get('facts', {})

# US-GAAP namespace contains most financial data
us_gaap = facts.get('us-gaap', {})
# all available metrics in the us-gaap namespace
available_metrics = list(us_gaap.keys())
print(f"Available metrics for CIK {cik_padded}: {available_metrics}")

# all available units
available_units = set()
for metric, metric_data in us_gaap.items():
    for unit in metric_data.get('units', {}):
        available_units.add(unit)
print(f"Available units for CIK {cik_padded}: {available_units}")

Available metrics for CIK 0001627475: ['AccountsPayableCurrent', 'AccountsReceivableGrossCurrent', 'AccountsReceivableNetCurrent', 'AccretionAmortizationOfDiscountsAndPremiumsInvestments', 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment', 'AdditionalPaidInCapitalCommonStock', 'AdjustmentsToAdditionalPaidInCapitalEquityComponentOfConvertibleDebtSubsequentAdjustments', 'AdjustmentsToAdditionalPaidInCapitalSharebasedCompensationRequisiteServicePeriodRecognitionValue', 'AdjustmentsToAdditionalPaidInCapitalStockIssuedIssuanceCosts', 'AdjustmentsToAdditionalPaidInCapitalWarrantIssued', 'AdvertisingExpense', 'AllocatedShareBasedCompensationExpense', 'AllowanceForDoubtfulAccountsReceivable', 'AllowanceForDoubtfulAccountsReceivableCurrent', 'AllowanceForDoubtfulAccountsReceivableWriteOffs', 'AmortizationOfFinancingCosts', 'AmortizationOfIntangibleAssets', 'AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount', 'AssetImpairmentCharges', 'Assets', 'Asset

In [3]:
concepts_to_fetch = ['Revenues', 'NetIncomeLoss', 'OperatingExpenses',
                     'Assets', 'Liabilities', 'RevenueFromContractWithCustomerExcludingAssessedTax', 
                     'EarningsPerShareDiluted', 'CommonStockSharesOutstanding',
                     'WeightedAverageNumberOfSharesOutstandingBasic']

In [4]:
def fetch_tickers(url="", table=0, index_col="Symbol", company_col="Security",  cik_col="CIK"):
    # Fetch with proper User-Agent header to avoid 403 error
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    tables = pd.read_html(io.StringIO(response.text))
    index_table = tables[table]  # Use the specified table index
    tickers = index_table[index_col].tolist()
    companies = index_table[company_col].tolist() if company_col in index_table.columns else None
    # cik = index_table[cik_col].tolist() if cik_col in index_table.columns else Nones
    
    # dataframe of tickers, companies, cik
    df = pd.DataFrame({
        'Company': companies,
        'Ticker': tickers,
        # 'CIK': cik
    })

    return df

sp500_url = 'http://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

In [5]:
sp500_df = fetch_tickers(sp500_url)
# convert sp500_df to dictionary with company as key and ticker as value
sp500_tickers = dict(zip(sp500_df['Company'], sp500_df['Ticker']))
sp500_tickers

{'3M': 'MMM',
 'A. O. Smith': 'AOS',
 'Abbott Laboratories': 'ABT',
 'AbbVie': 'ABBV',
 'Accenture': 'ACN',
 'Adobe Inc.': 'ADBE',
 'Advanced Micro Devices': 'AMD',
 'AES Corporation': 'AES',
 'Aflac': 'AFL',
 'Agilent Technologies': 'A',
 'Air Products': 'APD',
 'Airbnb': 'ABNB',
 'Akamai Technologies': 'AKAM',
 'Albemarle Corporation': 'ALB',
 'Alexandria Real Estate Equities': 'ARE',
 'Align Technology': 'ALGN',
 'Allegion': 'ALLE',
 'Alliant Energy': 'LNT',
 'Allstate': 'ALL',
 'Alphabet Inc. (Class A)': 'GOOGL',
 'Alphabet Inc. (Class C)': 'GOOG',
 'Altria': 'MO',
 'Amazon': 'AMZN',
 'Amcor': 'AMCR',
 'Ameren': 'AEE',
 'American Electric Power': 'AEP',
 'American Express': 'AXP',
 'American International Group': 'AIG',
 'American Tower': 'AMT',
 'American Water Works': 'AWK',
 'Ameriprise Financial': 'AMP',
 'Ametek': 'AME',
 'Amgen': 'AMGN',
 'Amphenol': 'APH',
 'Analog Devices': 'ADI',
 'Aon plc': 'AON',
 'APA Corporation': 'APA',
 'Apollo Global Management': 'APO',
 'Apple Inc.

In [6]:
gig_tickers = {
    'Upwork': "UPWK",
    'Fiverr': "FVRR",
    'Uber': "UBER",
    'Lyft': "LYFT",
    'Doordash': "DASH",
    'Grubhub': "GRUB",
    'Instacart': "CART",
    'Etsy': "ETSY",
    'Shopify': "SHOP",
    'Udemy': "UDMY",
    'Herbalife': "HLF",
    'Primerica': "PRI",
    'Tupperware': "TUP",
    'Avon': "AVP",
    'Nu Skin': "NUS",
    'USANA': "USNA",  
}

# Get the CIK map - returns dict with "ticker" and "title" keys
cik_map = get_cik_map(user_agent='Academic Research Project (gig-economy-capstone@umich.edu)')

# Access ticker -> CIK mapping (keys are uppercase)
ticker_to_cik = cik_map["ticker"]

# Debug: check what keys look like
# sample_tickers = list(ticker_to_cik.keys())[:10]
# print(f"Sample tickers in CIK map: {sample_tickers}")

# Lookup CIKs for our tickers
gig_ciks = {}
for company, ticker in gig_tickers.items():
    lookup = ticker.upper()
    if lookup in ticker_to_cik:
        gig_ciks[ticker] = ticker_to_cik[lookup]
    else:
        print(f"Ticker {ticker} not found in CIK map")

print(f"\nFound CIKs: {gig_ciks}")

sp500_ciks = {}
for company, ticker in sp500_tickers.items():
    lookup = ticker.upper()
    if lookup in ticker_to_cik:
        sp500_ciks[ticker] = ticker_to_cik[lookup]
    else:
        print(f"Ticker {ticker} not found in CIK map")

print(f"\nFound CIKs for S&P 500: {len(sp500_ciks)} companies found")

Ticker GRUB not found in CIK map
Ticker TUP not found in CIK map
Ticker AVP not found in CIK map

Found CIKs: {'UPWK': '1627475', 'FVRR': '1762301', 'UBER': '1543151', 'LYFT': '1759509', 'DASH': '1792789', 'CART': '1579091', 'ETSY': '1370637', 'SHOP': '1594805', 'UDMY': '1607939', 'HLF': '1180262', 'PRI': '1475922', 'NUS': '1021561', 'USNA': '896264'}
Ticker BRK.B not found in CIK map
Ticker BF.B not found in CIK map

Found CIKs for S&P 500: 501 companies found


In [7]:
# recovered missing gig company CIKs by looking up tickers manually
# add missing gig company CIKs to gig_ciks dictionary
gig_ciks['GRUB'] = '0001594109'
gig_ciks['TUP'] = '0001008654'
gig_ciks['AVON'] = '0000008868'
gig_ciks

{'UPWK': '1627475',
 'FVRR': '1762301',
 'UBER': '1543151',
 'LYFT': '1759509',
 'DASH': '1792789',
 'CART': '1579091',
 'ETSY': '1370637',
 'SHOP': '1594805',
 'UDMY': '1607939',
 'HLF': '1180262',
 'PRI': '1475922',
 'NUS': '1021561',
 'USNA': '896264',
 'GRUB': '0001594109',
 'TUP': '0001008654',
 'AVON': '0000008868'}

In [8]:
# to a dictionary with ticker as key and dictionary of ticker and cik
gig_ciks_dict = {ticker: {"ticker": ticker, "cik": str(cik)} for ticker, cik in gig_ciks.items()}
sp500_ciks_dict = {ticker: {"ticker": ticker, "cik": str(cik)} for ticker, cik in sp500_ciks.items()}

In [9]:
def get_company_financials_xbrl(cik, metrics=None):
    if metrics is None:
        metrics = concepts_to_fetch
    
    cik_padded = cik.zfill(10)
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik_padded}.json"
    
    headers = {
        'User-Agent': 'Academic Research Project (gig-economy-capstone@umich.edu)',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        results = []
        facts = data.get('facts', {})
        
        # US-GAAP namespace contains most financial data
        us_gaap = facts.get('us-gaap', {})
        units = {'Segment', 'shares', 'segment', 'tradingDay', 'pure', 'USD/shares', 'USD'}
        for metric in metrics:
            if metric in us_gaap:
                data_items = []
                for unit in units:
                    if unit in us_gaap[metric].get('units', {}):
                        data_items.extend(us_gaap[metric]['units'][unit])
                for item in data_items:
                    if item.get('form') in ['10-K', '10-Q', '20-F', '6-K']:
                        results.append({
                            'form': item.get('form'),
                            'metric': metric,
                            'value': item.get('val'),
                            'fiscal_year': item.get('fy'),
                            'fiscal_period': item.get('fp'),
                            'filed_date': item.get('filed'),
                            'start_date': item.get('start'),
                            'end_date': item.get('end'),
                            # 'accession_number': item.get('accn'),
                        })
        
        df = pd.DataFrame(results)
        
        if not df.empty:
            # Pivot to get metrics as columns
            df_pivot = df.pivot_table(
                index=['filed_date', 'fiscal_year', 'form', 'fiscal_period'],
                columns='metric',
                values='value',
                aggfunc='first'
            ).reset_index()
            
            return df_pivot
        
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching XBRL data for CIK {cik}: {e}")
        return pd.DataFrame()


def get_all_companies_financials(companies=None):
    if companies is None:
        companies = gig_ciks_dict
    
    all_financials = []
    
    for company_name, company_info in companies.items():
        print(f"Fetching financials for {company_name}...")
        time.sleep(0.1)  # Rate limiting
        
        try:
            df = get_company_financials_xbrl(company_info['cik'])
            if not df.empty:
                # df['company'] = company_name
                df['ticker'] = company_info['ticker']
                # df['category'] = company_info['category']
                all_financials.append(df)
                print(f"  Found {len(df)} annual records")
            else:
                print(f"  No financial data found")
        except Exception as e:
            print(f"  Error: {e}")
            continue
    
    if all_financials:
        combined = pd.concat(all_financials, ignore_index=True)
        combined = combined.rename_axis(None, axis=1)
        return combined
    else:
        return pd.DataFrame()


In [10]:
gig_edgar_df_raw = get_all_companies_financials(gig_ciks_dict)
gig_edgar_df_raw = gig_edgar_df_raw[gig_edgar_df_raw['fiscal_year'] != 2026].reset_index(drop=True)
gig_edgar_df_raw.to_csv("../../data/edgar_sec/gig_edgar_sec_raw.csv", index=False)
# gig_edgar_df_raw = pd.read_csv("../../data/edgar_sec/gig_edgar_sec_raw.csv")
gig_edgar_df_raw

Fetching financials for UPWK...
  Found 30 annual records
Fetching financials for FVRR...
  Found 8 annual records
Fetching financials for UBER...
  Found 28 annual records
Fetching financials for LYFT...
  Found 28 annual records
Fetching financials for DASH...
  Found 21 annual records
Fetching financials for CART...
  Found 10 annual records
Fetching financials for ETSY...
  Found 44 annual records
Fetching financials for SHOP...
  Found 6 annual records
Fetching financials for UDMY...
  Found 18 annual records
Fetching financials for HLF...
  Found 63 annual records
Fetching financials for PRI...
  Found 59 annual records
Fetching financials for NUS...
  Found 63 annual records
Fetching financials for USNA...
  Found 59 annual records
Fetching financials for GRUB...
  Found 29 annual records
Fetching financials for TUP...
  Found 54 annual records
Fetching financials for AVON...
  Found 56 annual records


,filed_date,fiscal_year,form,fiscal_period,Assets,CommonStockSharesOutstanding,EarningsPerShareDiluted,Liabilities,NetIncomeLoss,OperatingExpenses,RevenueFromContractWithCustomerExcludingAssessedTax,WeightedAverageNumberOfSharesOutstandingBasic,ticker,Revenues
0,2018-11-08,2018,10-Q,Q3,2.751890e+08,3.374032e+07,NaN,1.400700e+08,1068000.0,98118000.0,1.477930e+08,NaN,UPWK,NaN
1,2019-03-07,2018,10-K,FY,2.751890e+08,3.374032e+07,NaN,1.400700e+08,-16233000.0,116335000.0,1.644450e+08,NaN,UPWK,NaN
2,2019-05-08,2019,10-Q,Q1,3.915730e+08,1.064543e+08,NaN,1.478280e+08,-6784000.0,45610000.0,5.921800e+07,NaN,UPWK,NaN
3,2019-08-07,2019,10-Q,Q2,3.915730e+08,1.064543e+08,NaN,1.478280e+08,-7196000.0,87505000.0,1.218990e+08,NaN,UPWK,NaN
4,2019-11-06,2019,10-Q,Q3,3.915730e+08,1.064543e+08,NaN,1.478280e+08,-14542000.0,134448000.0,1.860120e+08,NaN,UPWK,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,2022-05-06,2022,10-Q,Q1,2.307800e+09,1.013400e+02,NaN,3.162500e+09,-63600000.0,NaN,8.504000e+08,NaN,AVON,9.044000e+08
572,2022-08-12,2022,10-Q,Q2,2.307800e+09,1.013400e+02,NaN,3.162500e+09,-125200000.0,NaN,1.715200e+09,NaN,AVON,1.821400e+09
573,2022-11-10,2022,10-Q,Q3,2.307800e+09,1.013400e+02,NaN,3.162500e+09,-167700000.0,NaN,2.443900e+09,NaN,AVON,2.589700e+09
574,2023-03-14,2022,10-K,FY,2.564300e+09,1.013400e+02,NaN,3.162500e+09,-362800000.0,NaN,3.431200e+09,NaN,AVON,3.625200e+09


In [11]:
sp500_edgar_df_raw = get_all_companies_financials(sp500_ciks_dict)
sp500_edgar_df_raw = sp500_edgar_df_raw[sp500_edgar_df_raw['fiscal_year'] != 2026].reset_index(drop=True)
sp500_edgar_df_raw.to_csv("../../data/edgar_sec/sp500_edgar_sec_raw.csv", index=False)

# sp500_edgar_df_raw = pd.read_csv("../../data/edgar_sec/sp500_edgar_sec_raw.csv")

sp500_edgar_df_raw

Fetching financials for MMM...
  Found 67 annual records
Fetching financials for AOS...
  Found 62 annual records
Fetching financials for ABT...
  Found 67 annual records
Fetching financials for ABBV...
  Found 52 annual records
Fetching financials for ACN...
  Found 66 annual records
Fetching financials for ADBE...
  Found 68 annual records
Fetching financials for AMD...
  Found 63 annual records
Fetching financials for AES...
  Found 67 annual records
Fetching financials for AFL...
  Found 66 annual records
Fetching financials for A...
  Found 65 annual records
Fetching financials for APD...
  Found 67 annual records
Fetching financials for ABNB...
  Found 20 annual records
Fetching financials for AKAM...
  Found 65 annual records
Fetching financials for ALB...
  Found 63 annual records
Fetching financials for ARE...
  Found 62 annual records
Fetching financials for ALGN...
  Found 63 annual records
Fetching financials for ALLE...
  Found 48 annual records
Fetching financials for LNT

,filed_date,fiscal_year,form,fiscal_period,Assets,CommonStockSharesOutstanding,EarningsPerShareDiluted,Liabilities,NetIncomeLoss,RevenueFromContractWithCustomerExcludingAssessedTax,Revenues,WeightedAverageNumberOfSharesOutstandingBasic,ticker,OperatingExpenses
0,2009-07-31,2009,10-Q,Q2,2.579300e+10,NaN,2.70,1.548900e+10,1.933000e+09,NaN,NaN,704300000.0,MMM,NaN
1,2009-10-30,2009,10-Q,Q3,2.579300e+10,NaN,4.11,1.548900e+10,2.924000e+09,NaN,NaN,701300000.0,MMM,NaN
2,2010-02-16,2009,10-K,FY,2.579300e+10,693543287.0,5.60,1.548900e+10,4.096000e+09,NaN,NaN,718300000.0,MMM,NaN
3,2010-05-05,2010,10-Q,Q1,2.725000e+10,NaN,0.74,1.394800e+10,5.180000e+08,NaN,NaN,693500000.0,MMM,NaN
4,2010-08-04,2010,10-Q,Q2,2.725000e+10,NaN,1.86,1.394800e+10,1.301000e+09,NaN,NaN,695200000.0,MMM,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29442,2025-02-13,2024,10-K,FY,1.428600e+10,458367358.0,4.49,9.295000e+09,2.114000e+09,NaN,8.080000e+09,468891000.0,ZTS,NaN
29443,2025-05-06,2025,10-Q,Q1,1.423700e+10,448743073.0,1.31,9.467000e+09,5.990000e+08,NaN,2.190000e+09,458000000.0,ZTS,NaN
29444,2025-08-05,2025,10-Q,Q2,1.423700e+10,448743073.0,2.67,9.467000e+09,1.223000e+09,NaN,4.551000e+09,456700000.0,ZTS,NaN
29445,2025-11-04,2025,10-Q,Q3,1.423700e+10,448743073.0,4.18,9.467000e+09,1.905000e+09,NaN,6.939000e+09,455400000.0,ZTS,NaN


In [12]:
print(sp500_edgar_df_raw.isna().sum())

filed_date                                                 0
fiscal_year                                                0
form                                                       0
fiscal_period                                              0
Assets                                                    58
CommonStockSharesOutstanding                           12685
EarningsPerShareDiluted                                 1787
Liabilities                                             9221
NetIncomeLoss                                           3557
RevenueFromContractWithCustomerExcludingAssessedTax    20651
Revenues                                               15365
WeightedAverageNumberOfSharesOutstandingBasic           1949
ticker                                                     0
OperatingExpenses                                      22488
dtype: int64


In [13]:
# Feature engineering
def feature_engineering(df):
    df = df.copy()
    
    # if 'date' column doesn't exist, create it based on 'fiscal_period' and 'fiscal_year'
    if 'date' not in df.columns:
        fiscal_period_mapping = {
            'Q1': '03-31',
            'Q2': '06-30',
            'Q3': '09-30',
            'Q4': '12-31',
            'FY': '12-31'
        }
        
        df['date'] = df['fiscal_period'].map(fiscal_period_mapping).fillna(df['fiscal_period'])
        # use 'fiscal_year' and 'date' to create a proper datetime column
        df['date'] = df.apply(lambda row: f"{row['fiscal_year']}-{row['date']}" if pd.notna(row['fiscal_year']) and pd.notna(row['date']) else pd.NaT, axis=1)
    
    df = df.sort_values(['ticker', 'date'])
    
    # combine all Revenue-related columns into a single 'Revenue' column, preferring 'Revenues' if present
    revenue_cols = [col for col in df.columns if isinstance(col, str) and 'Revenue' in col and col != 'Revenue']
    if revenue_cols:
        # sort so 'Revenues' comes first (highest priority)
        ordered = sorted(revenue_cols, key=lambda c: (c != 'Revenues'))
        df['Revenue'] = df[ordered[0]]
        for col in ordered[1:]:
            df['Revenue'] = df['Revenue'].combine_first(df[col])
        df = df.drop(columns=revenue_cols, errors='ignore')
        
    # fill missing NetIncomeLoss row-by-row where Revenue and OperatingExpenses are available
    if 'Revenue' in df.columns and 'OperatingExpenses' in df.columns:
        df['OperatingExpenses'] = df['OperatingExpenses'].fillna(0)
        calculated = df['Revenue'] - df['OperatingExpenses']
        if 'NetIncomeLoss' in df.columns:
            df['NetIncomeLoss'] = df['NetIncomeLoss'].combine_first(calculated)
        else:
            df['NetIncomeLoss'] = calculated
    
    # combine all SharesOutstanding-related columns into a single 'SharesOutstanding' column, preferring 'WeightedAverageNumberOfSharesOutstandingBasic' if present
    shares_cols = [col for col in df.columns if isinstance(col, str) and 'SharesOutstanding' in col and col != 'SharesOutstanding']
    if shares_cols:
        # sort so 'WeightedAverageNumberOfSharesOutstandingBasic' comes first (highest priority)
        ordered = sorted(shares_cols, key=lambda c: (c != 'WeightedAverageNumberOfSharesOutstandingBasic'))
        df['SharesOutstanding'] = df[ordered[0]]
        for col in ordered[1:]:
            df['SharesOutstanding'] = df['SharesOutstanding'].combine_first(df[col])
        df = df.drop(columns=shares_cols, errors='ignore')
        
    # fill missing EarningsPerShare row-by-row where NetIncomeLoss and SharesOutstanding are available
    if 'NetIncomeLoss' in df.columns and 'SharesOutstanding' in df.columns:
        shares = df['SharesOutstanding'].replace(0, pd.NA)
        calculated_eps = df['NetIncomeLoss'] / shares
        if 'EarningsPerShareDiluted' in df.columns:
            df['EarningsPerShareDiluted'] = df['EarningsPerShareDiluted'].combine_first(calculated_eps)
        else:
            df['EarningsPerShareDiluted'] = calculated_eps
    
    df['Quarter'] = df['fiscal_period'].apply(lambda x: 4 if x == 'FY' else int(x[1]) if pd.notna(x) and x.startswith('Q') else None)
    
    # calculate NetWorth from Assets and Liabilities
    # fill Liabilities with 0 if it's missing but Assets is present, and vice versa, to allow NetWorth calculation
    df['Assets'] = df['Assets'].fillna(0)
    df['Liabilities'] = df['Liabilities'].fillna(0)
    df['NetWorth'] = df['Assets'] - df['Liabilities']
    
    # calculate SharesOutstanding from WeightedAverageNumberOfSharesOutstandingBasic by adjusting for quarter length (assuming basic shares are reported as a weighted average over the fiscal period)
    df['SharesOutstanding'] = df['SharesOutstanding'].abs()
    df['SharesOutstanding'] = df['SharesOutstanding'] * (12/(df['Quarter'] * 3))
    
    # drop Quarter column
    df = df.drop(columns=['Quarter', 'Revenue', 'WeightedAverageNumberOfSharesOutstandingBasic', 'Assets', 'Liabilities', 'OperatingExpenses'], errors='ignore')
        
    # fill nan values per ticker with mean + std * N(0,1) noise for each numeric metric
    numeric_cols = df.select_dtypes(include='number').columns.difference(['fiscal_year'])
    for metric in numeric_cols:
        def fill_with_noise(x):
            n_missing = x.isna().sum()
            if n_missing == 0:
                return x
            mu = x.mean()
            sigma = x.std()
            if pd.isna(mu):
                return x  # no observed values to draw from — leave as NaN
            noise = mu + (sigma if pd.notna(sigma) and sigma > 0 else 0) * np.random.randn(n_missing)
            x = x.copy()
            x[x.isna()] = noise
            return x
        df[metric] = df.groupby('ticker')[metric].transform(fill_with_noise)
    
    # ticker to 1st column
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('ticker')))
    cols.insert(1, cols.pop(cols.index('date')))
    df = df[cols]
    
    # limit date range to 2009-2025
    df = df[(df['fiscal_year'] >= 2009) & (df['fiscal_year'] <= 2025)]
    
    return df.reset_index(drop=True)


In [14]:
gig_edgar_copy = gig_edgar_df_raw.copy()

gig_edgar_df = feature_engineering(gig_edgar_copy)

gig_edgar_df.to_csv("../../data/edgar_sec/gig_edgar_sec.csv", index=False)

print(gig_edgar_df.isna().sum())
gig_edgar_df

ticker                     0
date                       0
filed_date                 0
fiscal_year                0
form                       0
fiscal_period              0
EarningsPerShareDiluted    0
NetIncomeLoss              0
SharesOutstanding          0
NetWorth                   0
dtype: int64


,ticker,date,filed_date,fiscal_year,form,fiscal_period,EarningsPerShareDiluted,NetIncomeLoss,SharesOutstanding,NetWorth
0,AVON,2009-06-30,2009-07-30,2009,10-Q,Q2,0.97,420300000.0,1.093611e+09,7.123000e+08
1,AVON,2009-09-30,2009-10-29,2009,10-Q,Q3,1.49,642900000.0,4.662768e+08,7.123000e+08
2,AVON,2009-12-31,2010-02-25,2009,10-K,FY,1.21,530700000.0,4.334700e+08,7.123000e+08
3,AVON,2010-03-31,2010-04-30,2010,10-Q,Q1,0.27,117300000.0,2.119975e+08,1.312600e+09
4,AVON,2010-06-30,2010-07-29,2010,10-Q,Q2,0.46,200200000.0,8.532800e+08,1.312600e+09
...,...,...,...,...,...,...,...,...,...,...
571,USNA,2024-09-30,2024-11-05,2024,10-Q,Q3,2.43,47022000.0,2.571067e+07,6.327570e+08
572,USNA,2025-03-31,2025-05-06,2025,10-Q,Q1,0.86,16537000.0,7.669600e+07,5.863320e+08
573,USNA,2025-06-30,2025-08-05,2025,10-Q,Q2,1.40,26969000.0,3.824600e+07,5.863320e+08
574,USNA,2025-09-30,2025-11-05,2025,10-Q,Q3,1.96,37576000.0,2.547733e+07,5.863320e+08


In [15]:
sp500_edgar_copy = sp500_edgar_df_raw.copy()

sp500_edgar_df = feature_engineering(sp500_edgar_copy)

sp500_edgar_df.to_csv("../../data/edgar_sec/sp500_edgar_sec.csv", index=False)

print(sp500_edgar_df.isna().sum())
sp500_edgar_df

ticker                       0
date                         0
filed_date                   0
fiscal_year                  0
form                         0
fiscal_period                0
EarningsPerShareDiluted    761
NetIncomeLoss               65
SharesOutstanding          248
NetWorth                     0
dtype: int64


,ticker,date,filed_date,fiscal_year,form,fiscal_period,EarningsPerShareDiluted,NetIncomeLoss,SharesOutstanding,NetWorth
0,A,2009-12-31,2009-12-21,2009,10-K,FY,1.57,4.288339e+08,3.940000e+02,2.559000e+09
1,A,2010-03-31,2010-03-10,2010,10-Q,Q1,0.18,1.246776e+09,1.404000e+03,2.514000e+09
2,A,2010-06-30,2010-06-07,2010,10-Q,Q2,-0.11,1.006710e+09,6.960000e+02,2.514000e+09
3,A,2010-12-31,2010-12-20,2010,10-K,FY,1.87,9.064356e+08,3.630000e+08,2.514000e+09
4,A,2011-03-31,2011-03-09,2011,10-Q,Q1,0.22,2.189663e+09,1.392000e+03,3.236000e+09
...,...,...,...,...,...,...,...,...,...,...
29440,ZTS,2024-12-31,2025-02-13,2024,10-K,FY,4.49,2.114000e+09,4.688910e+08,4.991000e+09
29441,ZTS,2025-03-31,2025-05-06,2025,10-Q,Q1,1.31,5.990000e+08,1.832000e+09,4.770000e+09
29442,ZTS,2025-06-30,2025-08-05,2025,10-Q,Q2,2.67,1.223000e+09,9.134000e+08,4.770000e+09
29443,ZTS,2025-09-30,2025-11-04,2025,10-Q,Q3,4.18,1.905000e+09,6.072000e+08,4.770000e+09
